# RideBase 08 — V1 Target Engineering Experiments

Bu notebook **RideBase Synthetic Dataset v1.2** üzerinde, sonraki servis tahmini için
farklı **target (hedef değişken) tanımlarını** karşılaştırır. Model ailesini daha fazla
zorlamak yerine hedefi farklı biçimlerde tanımlar ve şu soruyu araştırır:

> **"Model mi yetersiz, yoksa hedef fazla mı gürültülü?"**

06 (V1 Basic) ve 07 (V1 Advanced) DAYS R² değerleri birbirine yakındır; feature
engineering / tuning / ensemble yalnızca küçük iyileşme sağladı. Bu notebook target
tanımını değiştirerek gerçek tavanı ölçer. **Production validation hâlâ `BLOCKED`.**

### Yöntem disiplini
- **Feature matrisi sabit**: yalnız 05 preprocessor'ın ürettiği `SET_A_BASE_05` (277 encoded,
  TRAIN-fit). Böylece skordaki fark **target tanımından** gelir, feature'dan değil.
- **Model sabit**: target-to-target kıyasında `HistGradientBoostingRegressor` (default).
- **Split disiplini**: authoritative `TRAIN / VALIDATION / TEST`; random split yok. Target
  formülasyonu seçimi yalnız TRAIN+VALIDATION üzerinden; TEST sadece final kıyas.
- **Leakage yasağı**: geleceğe bakış yalnız *target tanımlamak* için kullanılabilir,
  asla feature olarak değil. R²'yi yükseltmek için target uydurma / outlier temizliği /
  plansız servisleri sessizce silme **yapılmaz.**

### Kavramlar (basit Türkçe)
| kavram | anlamı |
|---|---|
| **target engineering** | hedef değişkeni farklı/anlamlı biçimlerde yeniden tanımlamak |
| **target noise** | hedefin, feature'larla açıklanamayan rastgele oynaması |
| **residual learning** | ham hedef yerine "kural baseline ne kadar yanıldı?"ı öğrenmek |
| **normalized / ratio target** | hedefi bir referansa bölmek (ör. gerçek interval ÷ politika interval) |
| **log transform** | sağ-çarpık hedefi `log1p` uzayında öğrenip `expm1` ile geri çevirmek |
| **stochastic target** | üretim sürecinde rastgelelik içeren hedef (arıza zamanlaması gibi) |
| **planned maintenance** | planlı periyodik bakım (`PERIODIC`) |
| **unscheduled service** | plansız servis (`REPAIR` / `BREAKDOWN` / `TIRE`) |
| **target-definition bias** | "skor güzel çıktı" diye yanlış hedefi seçme hatası |


In [ ]:
from pathlib import Path
from collections import OrderedDict
import json, platform, warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn import __version__ as sklearn_version
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None
try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None
try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 160)

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "models").is_dir():
            return candidate
    raise FileNotFoundError("ridebase-ml proje koku bulunamadi")

ROOT = find_project_root()
DATASET_ROOT = ROOT.parent / "ridebase_v1_2"
SOURCE = DATASET_ROOT / "source_tables"
DERIVED = DATASET_ROOT / "derived_outputs"
MODELS = ROOT / "models"; OUTPUTS = ROOT / "outputs"; REPORTS = ROOT / "reports"
TABLES = REPORTS / "tables"; FIGURES = REPORTS / "figures" / "v1_target_engineering"
for d in [MODELS, OUTPUTS, TABLES, FIGURES]:
    d.mkdir(parents=True, exist_ok=True)
SEED = 42; DATASET_VERSION = "1.2.0"; MIN_SEGMENT_N = 50

DAYS_TOL = [15, 30, 45, 60, 90]
KM_TOL = [500, 1000, 1500, 2000, 5000]

def regression_metrics(y, pred, target):
    y = np.asarray(y, float); pred = np.asarray(pred, float); ae = np.abs(pred - y)
    out = {"mae": mean_absolute_error(y, pred), "median_ae": median_absolute_error(y, pred),
           "rmse": mean_squared_error(y, pred) ** 0.5, "r2": r2_score(y, pred),
           "bias": float(np.mean(pred - y)), "p90_ae": float(np.quantile(ae, .90))}
    for x in (DAYS_TOL if target == "DAYS" else KM_TOL):
        out[f"within_{x}"] = float(np.mean(ae <= x))
    return out

def noise_stats(values):
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    mean = float(np.mean(v)); std = float(np.std(v))
    return {"n": int(v.size), "mean": mean, "median": float(np.median(v)), "std": std,
            "variance": float(np.var(v)), "skewness": float(pd.Series(v).skew()),
            "coef_of_variation": float(std / mean) if mean else np.nan,
            "p95": float(np.quantile(v, .95)), "p99": float(np.quantile(v, .99)), "max": float(np.max(v))}

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGURES / name, dpi=150, bbox_inches="tight"); plt.close()

plt.style.use("seaborn-v0_8-whitegrid")
print("SETUP OK", "xgb", XGBRegressor is not None, "lgbm", LGBMRegressor is not None, "catboost", CatBoostRegressor is not None)

## 1 — Dataset version guard & artifact yükleme

**Ne yapıyoruz?** Yalnız `dataset_version == generator_version == 1.2.0` doğrulanır; 05
preprocessor, 06 basic modelleri, 07 advanced metrikleri ve 04 V0 tahminleri yüklenir.

**Neden?** Eski v1 / v1.1 çıktıları farklı üretim kurallarına sahiptir; karışırsa kıyas
geçersiz olur. `joined` = manifest ⋈ next-service targets ⋈ snapshots ⋈ split label cutoff.

**Observed-only contract:** V1 regression ailesi yalnız *gözlemlenen* sonraki servis
hedeflerini kullanır (`v1_regression_eligible`). Censored kayıtlara uydurma hedef yazılmaz.
Beklenen: TRAIN 20 679 · VALIDATION 1 932 · TEST 2 622.


In [ ]:
with open(DERIVED / "dataset_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)
info = metadata["dataset"]
if info.get("dataset_version") != DATASET_VERSION or info.get("generator_version") != DATASET_VERSION:
    raise RuntimeError("Yalniz RideBase Synthetic Dataset v1.2.0 kullanilabilir")

snapshots = pd.read_parquet(DERIVED / "ml_maintenance_snapshots.parquet")
targets = pd.read_parquet(DERIVED / "ml_next_service_targets.parquet")
manifest = pd.read_parquet(OUTPUTS / "ml_modeling_manifest.parquet")
split_manifest = pd.read_csv(DERIVED / "split_manifest.csv", encoding="utf-8-sig", low_memory=False)
base_preprocessor = joblib.load(MODELS / "ml_preprocessor_v1_2.joblib")
basic_days_model = joblib.load(MODELS / "v1_next_service_days_model.joblib")
basic_km_model = joblib.load(MODELS / "v1_next_service_km_model.joblib")
basic_metrics = pd.read_csv(TABLES / "v1_regression_metrics.csv")
advanced_metrics = pd.read_csv(TABLES / "v1_advanced_regression_metrics.csv")
v0_v1_basic = pd.read_csv(TABLES / "v0_vs_v1_regression_comparison.csv")
v0_predictions = pd.read_parquet(OUTPUTS / "v0_rule_baseline_next_service_predictions.parquet")

expected_splits = {"TRAIN": 27428, "VALIDATION": 6399, "TEST": 7691}
if len(snapshots) != 41518 or manifest.split.value_counts().to_dict() != expected_splits:
    raise RuntimeError("Dataset veya split guard failed")
if len(base_preprocessor.feature_names_in_) != 127 or len(base_preprocessor.get_feature_names_out()) != 277:
    raise RuntimeError("05 preprocessing contract mismatch")
for frame, name in [(snapshots, "snapshots"), (targets, "targets"), (manifest, "manifest")]:
    if frame.snapshot_id.duplicated().any():
        raise RuntimeError(f"Duplicate snapshot_id: {name}")

tcols = targets[["snapshot_id", "target_event_observed", "days_to_next_service", "km_to_next_service",
                 "target_km_valid", "next_service_type_code", "next_service_is_breakdown"]]
cutoff = split_manifest[["snapshot_id", "primary_label_cutoff_at"]]
joined = (manifest.merge(tcols, on="snapshot_id", how="left", validate="one_to_one", suffixes=("_manifest", "_target"))
                  .merge(snapshots, on=["snapshot_id", "motorcycle_id"], how="left", validate="one_to_one", suffixes=("_man", "_snap"))
                  .merge(cutoff, on="snapshot_id", how="left", validate="one_to_one"))
joined["snapshot_at"] = pd.to_datetime(joined["snapshot_at_man"])
joined["primary_label_cutoff_at"] = pd.to_datetime(joined["primary_label_cutoff_at"])
joined["split"] = joined["split"].astype(str)
joined = joined.set_index("snapshot_id", drop=False)
joined.index.name = "_snapshot_index"

DAYS_ANY = "days_to_next_service_target"
KM_ANY = "km_to_next_service_target"
KM_VALID = "target_km_valid_target"

days_any_mask = (joined.v1_regression_eligible.astype(bool) & joined[DAYS_ANY].notna()
                 & np.isfinite(joined[DAYS_ANY]) & (joined[DAYS_ANY] >= 0))
km_any_mask = (days_any_mask & joined[KM_ANY].notna() & np.isfinite(joined[KM_ANY])
               & (joined[KM_ANY] >= 0) & (joined[KM_VALID] == 1))
days_any_counts = joined.loc[days_any_mask].groupby("split").size().to_dict()
km_any_counts = joined.loc[km_any_mask].groupby("split").size().to_dict()
if days_any_counts != {"TRAIN": 20679, "VALIDATION": 1932, "TEST": 2622}:
    raise RuntimeError(f"Observed any-service days contract mismatch: {days_any_counts}")
print("DATASET_ARTIFACT_GUARD=PASS", info["dataset_version"])
print("OBSERVED_ANY_DAYS=", days_any_counts, "VALID_ANY_KM=", km_any_counts)

## 2 — Sabit feature matrisi (`SET_A_BASE_05`)

**Ne yapıyoruz?** 05 preprocessor'ı tüm satırlara uygulayıp 277 sütunlu yoğun bir matris
üretiyoruz. Bu notebook boyunca **tek feature seti** budur.

**Neden?** Target formülasyonları arasındaki farkın kaynağını izole etmek için feature'ları
dondurmak gerekir. 07'deki engineered feature'lar burada bilinçli olarak **kullanılmaz.**


In [ ]:
base_raw_features = list(base_preprocessor.feature_names_in_)
X_base_sparse = base_preprocessor.transform(joined[base_raw_features])
if not sparse.issparse(X_base_sparse) or not np.isfinite(X_base_sparse.data).all():
    raise RuntimeError("05 base transform QA failed")
X_base = X_base_sparse.toarray().astype("float32", copy=False)
print("X_base", X_base.shape)

SPLITS = ["TRAIN", "VALIDATION", "TEST"]
split_arr = joined.split.to_numpy()
def split_positions(mask_series):
    m = mask_series.to_numpy() if hasattr(mask_series, "to_numpy") else np.asarray(mask_series)
    return {s: np.flatnonzero(m & (split_arr == s)) for s in SPLITS}

any_days_pos = split_positions(days_any_mask)
any_km_pos = split_positions(km_any_mask)
any_obs_n = {s: len(any_days_pos[s]) for s in SPLITS}

## 3 — A) RAW TARGET (referans)

**Bu target neyi ifade ediyor?** `actual_next_service_days` / `actual_next_service_km_delta`
— motor **herhangi bir nedenle** (periyodik + plansız) bir sonraki kez servise geldiğinde
geçen gün / km. 06 ve 07'nin hedefi budur.

**Leakage riski?** Yok — hedef ayrı target tablosundan gelir, feature snapshot'ında ham
gelecek alanı yoktur.

**Sonuç nasıl yorumlanmalı?** Dağılım sağ-çarpıktır (birkaç çok uzun aralık). Yüksek
skewness ve yüksek varyasyon katsayısı (CoV), hedefin doğal gürültüsünün göstergesidir.


In [ ]:
y_any_days = joined[DAYS_ANY].to_numpy(float)
y_any_km = joined[KM_ANY].to_numpy(float)

raw_rows = []
for tgt, arr, pos in [("DAYS", y_any_days, any_days_pos), ("KM", y_any_km, any_km_pos)]:
    for split in SPLITS:
        raw_rows.append({"target": tgt, "split": split, **noise_stats(arr[pos[split]])})
raw_target_summary = pd.DataFrame(raw_rows)
print(raw_target_summary.to_string(index=False))

for tgt, arr, pos, fname in [("DAYS", y_any_days, any_days_pos, "01_target_distribution_raw_days.png"),
                             ("KM", y_any_km, any_km_pos, "02_target_distribution_raw_km.png")]:
    tr = arr[pos["TRAIN"]]
    plt.figure(figsize=(8, 4))
    plt.hist(tr[tr <= np.quantile(tr, .99)], bins=60, color="#4c78a8")
    plt.xlabel(f"actual_next_service_{tgt.lower()} (<=P99)"); plt.ylabel("count")
    plt.title(f"RAW next ANY service {tgt} target (TRAIN, skew={pd.Series(tr).skew():.2f})")
    savefig(fname)

## 4 — F) NEXT PERIODIC MAINTENANCE target (bu notebook'un kritik deneyi)

**Ne yapıyoruz?** `services.csv`'den her snapshot için `snapshot_at` sonrasındaki **ilk
`service_type_code == PERIODIC`** servisi bulup `days_to_next_periodic` ve
`km_to_next_periodic` hesaplıyoruz.

**Bu target neyi ifade ediyor?**
- *Next Any Service*: "Motor herhangi bir nedenle ne zaman tekrar gelir?"
- *Next Periodic Maintenance*: "Motorun **planlı bakımı** ne zaman gelir?"

**Leakage riski?** Target oluştururken geleceğe bakmak **normaldir** (bu hedefin kendisidir).
Bu bilgi feature setine girmez; leakage audit bunu açıkça ayırır.

**Censoring:** Snapshot sonrası split label cutoff'a kadar PERIODIC servis yoksa
`censored_periodic = 1`. V1 regression yalnız *observed* periodic hedefleri kullanır;
censored satırlara 0 / -1 yazılmaz. Coverage (any-observed'a oran) raporlanır.

**Ürün açısından anlamı:** GitHub #579 hedefi planlı bakım hatırlatmasıysa periodic target
daha doğru problem tanımı olabilir. **Skor güzel diye otomatik periodic seçilmez.**


In [ ]:
sv = pd.read_csv(SOURCE / "services.csv", low_memory=False)[
    ["motorcycle_id", "service_type_code", "received_at", "odometer_km"]]
sv["received_at"] = pd.to_datetime(sv.received_at)
periodic_sv = sv[sv.service_type_code == "PERIODIC"].sort_values(["motorcycle_id", "received_at"])
periodic_groups = {m: (g.received_at.to_numpy("datetime64[ns]"), g.odometer_km.to_numpy(float))
                   for m, g in periodic_sv.groupby("motorcycle_id")}

n = len(joined)
per_days = np.full(n, np.nan); per_km = np.full(n, np.nan)
per_at = np.full(n, np.datetime64("NaT"), "datetime64[ns]")
snap_at = joined.snapshot_at.to_numpy("datetime64[ns]")
snap_odo = pd.to_numeric(joined.snapshot_odometer_km, errors="coerce").to_numpy(float)
mc_id = joined.motorcycle_id.to_numpy()
for i in np.flatnonzero(days_any_mask.to_numpy()):
    g = periodic_groups.get(mc_id[i])
    if g is None:
        continue
    times, odo = g
    p = np.searchsorted(times, snap_at[i], side="right")
    if p >= len(times):
        continue
    per_at[i] = times[p]
    per_days[i] = (times[p] - snap_at[i]) / np.timedelta64(1, "D")
    per_km[i] = odo[p] - snap_odo[i]

cutoff_at = joined.primary_label_cutoff_at.to_numpy("datetime64[ns]")
periodic_found = ~pd.isna(per_at)
periodic_observed = periodic_found & (per_at <= cutoff_at) & (per_days >= 0)
periodic_censored = days_any_mask.to_numpy() & ~periodic_observed

per_days_mask = pd.Series(days_any_mask.to_numpy() & periodic_observed, index=joined.index)
per_km_mask = pd.Series(per_days_mask.to_numpy() & np.isfinite(per_km) & (per_km > 0), index=joined.index)
per_days_pos = split_positions(per_days_mask)
per_km_pos = split_positions(per_km_mask)

coverage_rows = []
for split in SPLITS:
    a = any_obs_n[split]
    coverage_rows.append({"target_family": "NEXT_PERIODIC_MAINTENANCE", "split": split,
                          "any_service_observed": a,
                          "periodic_observed": len(per_days_pos[split]),
                          "periodic_censored": int((periodic_censored & (split_arr == split)).sum()),
                          "coverage_vs_any": len(per_days_pos[split]) / a})
coverage_table = pd.DataFrame(coverage_rows)
print(coverage_table.to_string(index=False))

## 5 — Target formülasyon matrisi + leakage ilkesi

Her formülasyon için: (1) hangi olayı hedeflediği, (2) fit uzayı, (3) **inference'ta orijinal
gün/km ölçeğine nasıl geri döndüğü**, (4) eligibility maskesi tanımlanır. Tüm metrikler
**orijinal ölçekte**, ilgili ground-truth'a karşı hesaplanır (ANY formülasyonları any-service
gerçeğine, PERIODIC formülasyonları periodic gerçeğine karşı).

| formülasyon | fit hedefi | geri dönüş (reconstruct) |
|---|---|---|
| `RAW_ANY` | `actual` | `pred` |
| `LOG_ANY` | `log1p(actual)` | `expm1(pred)` |
| `V0_RESIDUAL_ANY` | `actual − v0_pred` | `v0_pred + pred` |
| `POLICY_RATIO_ANY` | `actual ÷ policy_expected` | `pred × policy_expected` |
| `HISTORY_DEVIATION_ANY` | `actual − hist_median` | `hist_median + pred` |
| `RAW_PERIODIC` / `LOG_PERIODIC` / `V0_RESIDUAL_PERIODIC` | yukarıdakiler, periodic gerçeğiyle | — |

**Leakage ilkesi:** V0 tahmini ve politika interval'i **snapshot anında bilinen** politika
hesaplarıdır (gerçek gelecek servis değil). Geçmiş ortalama interval bir snapshot feature'ıdır.
Hiçbir formülasyonda gerçek gelecek servis bilgisi feature olarak kullanılmaz →
`v1_target_leakage_audit.csv`.


In [ ]:
va = v0_predictions.set_index("snapshot_id").reindex(joined.index)
v0_days = pd.to_numeric(va.predicted_days_to_next_service, errors="coerce").to_numpy(float)
v0_km = pd.to_numeric(va.predicted_km_to_next_service, errors="coerce").to_numpy(float)
v0_days_expected = pd.to_numeric(va.raw_predicted_days_to_next_service, errors="coerce").to_numpy(float)
v0_km_expected = pd.to_numeric(va.raw_predicted_km_to_next_service, errors="coerce").to_numpy(float)
v0_available = va.prediction_available.fillna(False).to_numpy(bool)
hist_days = pd.to_numeric(joined.avg_service_interval_days, errors="coerce").to_numpy(float)
hist_km = pd.to_numeric(joined.avg_service_interval_km, errors="coerce").to_numpy(float)
seq = pd.to_numeric(joined.service_sequence, errors="coerce").to_numpy(float)

ident = lambda p, c: p
identity = {"fit": lambda y, c: y, "back": ident}
logt = {"fit": lambda y, c: np.log1p(y), "back": lambda p, c: np.expm1(p)}

def resid_spec(anchor):
    return {"fit": lambda y, c: y - c[anchor], "back": lambda p, c: c[anchor] + p}
def ratio_spec(anchor):
    return {"fit": lambda y, c: y / c[anchor], "back": lambda p, c: p * c[anchor]}

FORMULATIONS = OrderedDict()
def add_form(name, family, days_mask, km_mask, spec, ctx_days, ctx_km, business, future_to_define, future_as_feature="NONE"):
    FORMULATIONS[name] = {"family": family, "days_mask": days_mask, "km_mask": km_mask,
                          "spec": spec, "ctx_days": ctx_days, "ctx_km": ctx_km, "business": business,
                          "future_to_define": future_to_define, "future_as_feature": future_as_feature}

any_d = days_any_mask.to_numpy(); any_k = km_any_mask.to_numpy()
per_d = per_days_mask.to_numpy(); per_k = per_km_mask.to_numpy()

add_form("RAW_ANY", "ANY", pd.Series(any_d, index=joined.index), pd.Series(any_k, index=joined.index),
         identity, {}, {}, "Motor herhangi bir nedenle (periyodik + plansiz) ne zaman servise gelecek?",
         "actual_next_service timing/mileage")
add_form("LOG_ANY", "ANY", pd.Series(any_d, index=joined.index), pd.Series(any_k, index=joined.index),
         logt, {}, {}, "RAW_ANY ile ayni olay; sag-carpik hedef log1p uzayinda ogreniliyor.",
         "actual_next_service timing/mileage")
add_form("V0_RESIDUAL_ANY", "ANY",
         pd.Series(any_d & v0_available & np.isfinite(v0_days), index=joined.index),
         pd.Series(any_k & v0_available & np.isfinite(v0_km), index=joined.index),
         resid_spec("anchor"), {"anchor": v0_days}, {"anchor": v0_km},
         "Kural bazli V0 tahmininin ne kadar erken/gec oldugunu (residual) ogren.",
         "actual_next_service; V0 anchor snapshot-time policy hesabi (gelecek degil)")
add_form("POLICY_RATIO_ANY", "ANY",
         pd.Series(any_d & v0_available & np.isfinite(v0_days_expected) & (v0_days_expected > 7), index=joined.index),
         pd.Series(any_k & v0_available & np.isfinite(v0_km_expected) & (v0_km_expected > 100), index=joined.index),
         ratio_spec("anchor"), {"anchor": v0_days_expected}, {"anchor": v0_km_expected},
         "Gercek interval / politika beklenen interval oranini ogren (normalize hedef).",
         "actual_next_service; policy beklenen interval snapshot-time bilgisi")
add_form("HISTORY_DEVIATION_ANY", "ANY",
         pd.Series(any_d & (seq >= 3) & np.isfinite(hist_days) & (hist_days > 0), index=joined.index),
         pd.Series(any_k & (seq >= 3) & np.isfinite(hist_km) & (hist_km > 0), index=joined.index),
         resid_spec("anchor"), {"anchor": hist_days}, {"anchor": hist_km},
         "Motorun kendi gecmis ortalama interval'inden sapmayi ogren.",
         "actual_next_service; gecmis ortalama interval snapshot feature'i")
add_form("RAW_PERIODIC", "PERIODIC", pd.Series(per_d, index=joined.index), pd.Series(per_k, index=joined.index),
         identity, {}, {}, "Motorun planli (PERIODIC) bakimi ne zaman gelecek? Plansiz servis olay sayilmaz.",
         "first future PERIODIC service timing/mileage (target tanimi icin gelecege bakis normaldir)")
add_form("LOG_PERIODIC", "PERIODIC", pd.Series(per_d, index=joined.index), pd.Series(per_k, index=joined.index),
         logt, {}, {}, "RAW_PERIODIC ile ayni olay; log1p uzayinda ogreniliyor.",
         "first future PERIODIC service timing/mileage")
add_form("V0_RESIDUAL_PERIODIC", "PERIODIC",
         pd.Series(per_d & v0_available & np.isfinite(v0_days), index=joined.index),
         pd.Series(per_k & v0_available & np.isfinite(v0_km), index=joined.index),
         resid_spec("anchor"), {"anchor": v0_days}, {"anchor": v0_km},
         "V0 planli-bakim anchor'ina gore periodic residual ogren.",
         "first future PERIODIC service; V0 anchor snapshot-time policy hesabi")

leak_rows = []
for name, f in FORMULATIONS.items():
    leak_rows.append({"target_name": name, "target_family": f["family"],
                      "future_info_used_to_define_target": f["future_to_define"],
                      "future_info_used_as_feature": f["future_as_feature"],
                      "feature_matrix": "SET_A_BASE_05 (05 preprocessor, 277 encoded, TRAIN-fit)",
                      "leakage_status": "PASS" if f["future_as_feature"] == "NONE" else "REVIEW",
                      "notes": "Gelecek bilgisi yalniz target tanimlamak icin; feature olarak kullanilmadi."})
leakage_audit = pd.DataFrame(leak_rows)
leakage_audit.to_csv(TABLES / "v1_target_leakage_audit.csv", index=False, encoding="utf-8-sig")
if not leakage_audit.leakage_status.eq("PASS").all():
    raise RuntimeError("Target leakage audit failed")
print(leakage_audit[["target_name", "target_family", "leakage_status"]].to_string(index=False))

## 6 — Standardize model & değerlendirme yardımcısı

`eval_formulation` bir formülasyonu verilen model fabrikasıyla TRAIN∩eligible üzerinde eğitir,
istenen split'lerde tahmin eder, formülasyona özgü `back()` ile **orijinal ölçeğe** döndürür,
`clip(0, None)` uygular ve metrikleri ilgili ground-truth'a karşı hesaplar. Coverage = split'te
eligible satır ÷ o split'in any-observed satır sayısı.


In [ ]:
TRUTH = {("ANY", "DAYS"): y_any_days, ("ANY", "KM"): y_any_km,
         ("PERIODIC", "DAYS"): per_days, ("PERIODIC", "KM"): per_km}

def eval_formulation(name, tgt, model_factory, use_splits=("VALIDATION",)):
    f = FORMULATIONS[name]
    mask = f["days_mask"] if tgt == "DAYS" else f["km_mask"]
    ctx_src = f["ctx_days"] if tgt == "DAYS" else f["ctx_km"]
    pos = split_positions(mask)
    truth = TRUTH[(f["family"], tgt)]
    spec = f["spec"]
    tr = pos["TRAIN"]
    if len(tr) < 500:
        return None
    ctx_tr = {k: v[tr] for k, v in ctx_src.items()}
    y_fit = spec["fit"](truth[tr], ctx_tr)
    ok = np.isfinite(y_fit)
    model = model_factory()
    model.fit(X_base[tr][ok], y_fit[ok])
    results = {}
    for split in use_splits:
        sp = pos[split]
        ctx_sp = {k: v[sp] for k, v in ctx_src.items()}
        pred_t = np.asarray(model.predict(X_base[sp]), float)
        pred = np.clip(spec["back"](pred_t, ctx_sp), 0, None)
        finite = np.isfinite(pred)
        met = regression_metrics(truth[sp][finite], pred[finite], tgt)
        met["coverage_vs_any"] = len(sp) / any_obs_n[split]
        met["n"] = int(finite.sum())
        results[split] = met
    return {"model": model, "pos": pos, "results": results}

def hgb_factory():
    return HistGradientBoostingRegressor(random_state=SEED)

## 7 — Primary validation sweep (standardize model)

**Ne yapıyoruz?** 8 formülasyon × {DAYS, KM} için `HistGradientBoostingRegressor` (default)
ile VALIDATION metrikleri. **Seçim yalnız burada** yapılır; TEST açılmaz.

**Sonuç nasıl yorumlanmalı?** `val_r2` yüksek ama `coverage` düşükse (ör. `POLICY_RATIO`
yalnız birkaç yüz satır) bu formülasyon otomatik kazanan **değildir** — leaderboard'a girmek
için `coverage ≥ 0.80` şartı vardır.


In [ ]:
val_rows = []
primary_store = {}
for name in FORMULATIONS:
    for tgt in ["DAYS", "KM"]:
        out = eval_formulation(name, tgt, hgb_factory, use_splits=("TRAIN", "VALIDATION"))
        if out is None:
            continue
        primary_store[(name, tgt)] = out
        v = out["results"]["VALIDATION"]; t = out["results"]["TRAIN"]
        val_rows.append({"formulation": name, "family": FORMULATIONS[name]["family"], "target": tgt,
                         "model": "HistGradientBoostingRegressor(default)",
                         "train_rows": len(out["pos"]["TRAIN"]), "val_rows": v["n"],
                         "coverage": v["coverage_vs_any"],
                         "val_mae": v["mae"], "val_median_ae": v["median_ae"], "val_rmse": v["rmse"],
                         "val_r2": v["r2"], "val_bias": v["bias"], "train_r2": t["r2"],
                         "val_within_a": v[f"within_{(DAYS_TOL if tgt=='DAYS' else KM_TOL)[1]}"],
                         "val_within_b": v[f"within_{(DAYS_TOL if tgt=='DAYS' else KM_TOL)[3]}"]})
validation_results = pd.DataFrame(val_rows).sort_values(["target", "val_r2"], ascending=[True, False])
validation_results.to_csv(TABLES / "v1_target_engineering_validation_results.csv", index=False, encoding="utf-8-sig")
print(validation_results.to_string(index=False))

## 8 — Target noise diagnostics

Her formülasyonun **fit uzayındaki** hedefinin std / varyans / skewness / CoV değerleri.
`LOG` uzayında CoV düşer ama bu yapaydır (ölçek değişti) — orijinal ölçekteki R² ana metriktir.
`RESIDUAL` / `DEVIATION` hedefleri ham hedeften daha simetrik olabilir; bu, "daha kolay hedef"
demek zorunda değildir, TEST R² ile birlikte okunur.


In [ ]:
noise_rows = []
for name, f in FORMULATIONS.items():
    for tgt in ["DAYS", "KM"]:
        mask = f["days_mask"] if tgt == "DAYS" else f["km_mask"]
        ctx_src = f["ctx_days"] if tgt == "DAYS" else f["ctx_km"]
        pos = split_positions(mask)
        idx = np.concatenate([pos["TRAIN"], pos["VALIDATION"]])
        truth = TRUTH[(f["family"], tgt)]
        ctx = {k: v[idx] for k, v in ctx_src.items()}
        fit_target = f["spec"]["fit"](truth[idx], ctx)
        noise_rows.append({"formulation": name, "family": f["family"], "target": tgt,
                           "scale": "fit_space", **noise_stats(fit_target)})
noise_diagnostics = pd.DataFrame(noise_rows)
noise_diagnostics.to_csv(TABLES / "v1_target_noise_diagnostics.csv", index=False, encoding="utf-8-sig")
coverage_full = coverage_table.copy()
coverage_full.to_csv(TABLES / "v1_target_coverage.csv", index=False, encoding="utf-8-sig")
print(noise_diagnostics[["formulation", "target", "std", "skewness", "coef_of_variation"]].to_string(index=False))

## 9 — Secondary strong model check

En iyi 2 ANY formülasyonu (coverage ≥ 0.80) DAYS için ikinci güçlü model aileleriyle
(ExtraTrees, XGBoost, varsa LightGBM / CatBoost) tekrar test edilir. Amaç: **target
ranking'in model ailesine aşırı bağlı olup olmadığını** görmek. Devasa tuning yapılmaz.


In [ ]:
def factories():
    fac = {"ExtraTreesRegressor": lambda: ExtraTreesRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)}
    if XGBRegressor is not None:
        fac["XGBRegressor"] = lambda: XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                                   subsample=0.9, colsample_bytree=0.9, random_state=SEED,
                                                   n_jobs=-1, tree_method="hist", verbosity=0)
    if LGBMRegressor is not None:
        fac["LGBMRegressor"] = lambda: LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                                     random_state=SEED, n_jobs=-1, verbosity=-1)
    if CatBoostRegressor is not None:
        fac["CatBoostRegressor"] = lambda: CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                                             random_seed=SEED, verbose=0, allow_writing_files=False)
    return fac

top2 = (validation_results[(validation_results.target == "DAYS") & (validation_results.coverage >= 0.80)]
        .sort_values("val_r2", ascending=False).formulation.head(2).tolist())
sec_rows = []
for name in top2:  # DAYS only: sadece target ranking'in model ailesine bagimliligini gormek icin
    for fam, fac in factories().items():
        out = eval_formulation(name, "DAYS", fac, use_splits=("VALIDATION",))
        if out is None:
            continue
        v = out["results"]["VALIDATION"]
        sec_rows.append({"formulation": name, "target": "DAYS", "model": fam,
                         "val_mae": v["mae"], "val_r2": v["r2"], "coverage": v["coverage_vs_any"]})
    base = primary_store.get((name, "DAYS"))
    if base:
        v = base["results"]["VALIDATION"]
        sec_rows.append({"formulation": name, "target": "DAYS", "model": "HistGradientBoostingRegressor(default)",
                         "val_mae": v["mae"], "val_r2": v["r2"], "coverage": v["coverage_vs_any"]})
secondary_check = pd.DataFrame(sec_rows).sort_values(["formulation", "val_r2"], ascending=[True, False])
secondary_check.to_csv(TABLES / "v1_target_secondary_model_check.csv", index=False, encoding="utf-8-sig")
print("TOP2 FOR SECONDARY CHECK:", top2)
print(secondary_check.to_string(index=False))

## 10 — Any Service vs Periodic Maintenance + plansız olay gürültüsü

Next-ANY-service hedefinin ne kadarının plansız (`REPAIR` / `BREAKDOWN` / `TIRE`) olay
olduğunu ve bunların target varyansına etkisini raporlar. Bu **yalnız diagnostik** —
plansız olaylar RAW target'tan silinmez. Grafikler `reports/figures/v1_target_engineering/`
altına kaydedilir.


In [ ]:
def vr(form, tgt, col):
    q = validation_results[(validation_results.formulation == form) & (validation_results.target == tgt)]
    return float(q.iloc[0][col]) if len(q) else np.nan

avp_rows = []
for tgt in ["DAYS", "KM"]:
    for label, form in [("NEXT_ANY_SERVICE", "RAW_ANY"), ("NEXT_PERIODIC_MAINTENANCE", "RAW_PERIODIC")]:
        nz = noise_diagnostics[(noise_diagnostics.formulation == form) & (noise_diagnostics.target == tgt)].iloc[0]
        avp_rows.append({"target": tgt, "target_definition": label, "formulation": form,
                         "val_mae": vr(form, tgt, "val_mae"), "val_r2": vr(form, tgt, "val_r2"),
                         "coverage": vr(form, tgt, "coverage"),
                         "target_std": nz["std"], "target_cv": nz["coef_of_variation"], "target_skew": nz["skewness"]})
any_vs_periodic = pd.DataFrame(avp_rows)
any_vs_periodic.to_csv(TABLES / "v1_any_vs_periodic_comparison.csv", index=False, encoding="utf-8-sig")

unplanned = joined.loc[days_any_mask, "next_service_type_code"].value_counts()
unplanned_share = float(unplanned.reindex(["REPAIR", "BREAKDOWN", "TIRE"]).fillna(0).sum() / unplanned.sum())
print(any_vs_periodic.to_string(index=False))
print("UNPLANNED_EVENT_SHARE_IN_ANY_TARGET=", round(unplanned_share, 4), unplanned.to_dict())

for tgt, fname in [("DAYS", "03_target_formulation_days_r2.png"), ("KM", "04_target_formulation_km_r2.png")]:
    p = validation_results[validation_results.target == tgt].sort_values("val_r2")
    plt.figure(figsize=(9, 5)); plt.barh(p.formulation, p.val_r2, color="#4c78a8")
    plt.xlabel("Validation R2"); plt.title(f"{tgt} target formulation - Validation R2"); savefig(fname)
for tgt, fname in [("DAYS", "05_target_formulation_days_mae.png"), ("KM", "06_target_formulation_km_mae.png")]:
    p = validation_results[validation_results.target == tgt].sort_values("val_mae", ascending=False)
    plt.figure(figsize=(9, 5)); plt.barh(p.formulation, p.val_mae, color="#f58518")
    plt.xlabel("Validation MAE"); plt.title(f"{tgt} target formulation - Validation MAE"); savefig(fname)
for tgt, fname in [("DAYS", "07_coverage_vs_days_r2.png"), ("KM", "08_coverage_vs_km_r2.png")]:
    p = validation_results[validation_results.target == tgt]
    plt.figure(figsize=(7, 5)); plt.scatter(p.coverage, p.val_r2, s=60)
    for _, r in p.iterrows():
        plt.annotate(r.formulation, (r.coverage, r.val_r2), fontsize=7)
    plt.xlabel("coverage vs any-observed"); plt.ylabel("Validation R2")
    plt.title(f"{tgt} coverage vs R2 trade-off"); savefig(fname)
for tgt, fname in [("DAYS", "09_any_vs_periodic_days.png"), ("KM", "10_any_vs_periodic_km.png")]:
    p = any_vs_periodic[any_vs_periodic.target == tgt]
    plt.figure(figsize=(6, 4)); plt.bar(p.target_definition, p.val_r2, color=["#4c78a8", "#54a24b"])
    plt.ylabel("Validation R2"); plt.title(f"{tgt}: Any service vs Periodic maintenance"); savefig(fname)
for tgt, anchor, pos_any, fname in [("DAYS", v0_days, any_days_pos, "11_v0_residual_distribution_days.png"),
                                    ("KM", v0_km, any_km_pos, "12_v0_residual_distribution_km.png")]:
    idx = pos_any["TRAIN"]
    resid = (y_any_days if tgt == "DAYS" else y_any_km)[idx] - anchor[idx]
    resid = resid[np.isfinite(resid)]
    plt.figure(figsize=(8, 4))
    plt.hist(resid[(resid >= np.quantile(resid, .01)) & (resid <= np.quantile(resid, .99))], bins=60, color="#b279a2")
    plt.axvline(0, color="r", ls="--"); plt.xlabel(f"actual - V0_predicted ({tgt})"); plt.ylabel("count")
    plt.title(f"V0 residual distribution {tgt} (mean={np.mean(resid):.1f})"); savefig(fname)

## 11 — TEST değerlendirmesi (finalistler)

Validation'dan seçilen finalistler: `RAW_ANY` (referans) + en iyi 2 ANY formülasyonu +
`RAW_PERIODIC`. **TEST yalnız burada** açılır; TEST sonucuna bakıp yeni formülasyon seçilmez.
`val_to_test_r2_gap` genelleme kaybını gösterir.


In [ ]:
raw_any_val_r2 = {t: vr("RAW_ANY", t, "val_r2") for t in ["DAYS", "KM"]}
any_finalists = (validation_results[(validation_results.family == "ANY") & (validation_results.target == "DAYS")
                                    & (validation_results.coverage >= 0.80)]
                 .sort_values("val_r2", ascending=False).formulation.tolist())
finalists = list(dict.fromkeys((["RAW_ANY"] + any_finalists[:2] + ["RAW_PERIODIC"])))

test_rows = []
test_store = {}
for name in finalists:
    for tgt in ["DAYS", "KM"]:
        out = eval_formulation(name, tgt, hgb_factory, use_splits=("VALIDATION", "TEST"))
        if out is None:
            continue
        test_store[(name, tgt)] = out
        v = out["results"]["VALIDATION"]; te = out["results"]["TEST"]
        row = {"formulation": name, "family": FORMULATIONS[name]["family"], "target": tgt,
               "coverage": te["coverage_vs_any"], "val_r2": v["r2"], "test_mae": te["mae"],
               "test_median_ae": te["median_ae"], "test_rmse": te["rmse"], "test_r2": te["r2"],
               "test_bias": te["bias"], "val_to_test_r2_gap": v["r2"] - te["r2"]}
        for x in (DAYS_TOL if tgt == "DAYS" else KM_TOL):
            row[f"test_within_{x}"] = te[f"within_{x}"]
        test_rows.append(row)
test_results = pd.DataFrame(test_rows).sort_values(["target", "test_r2"], ascending=[True, False])
test_results.to_csv(TABLES / "v1_target_engineering_test_results.csv", index=False, encoding="utf-8-sig")
print(test_results.to_string(index=False))

def tr_metric(form, tgt, col):
    q = test_results[(test_results.formulation == form) & (test_results.target == tgt)]
    return float(q.iloc[0][col]) if len(q) else np.nan

## 12 — Final karar: hedef, tavan, V2 survival, verdict

**Karar kuralları (TEST'e göre değil, validation + coverage + iş anlamına göre):**
- Periodic DAYS TEST R² − RAW_ANY DAYS TEST R² ≥ 0.05 **ve** periodic coverage ≥ 0.90 →
  `NEXT_PERIODIC_MAINTENANCE`
- En iyi ANF formülasyonu RAW_ANY'yi DAYS R²'de ≥ 0.03 geçiyorsa → `NEXT_ANY_SERVICE` (o formülasyonla)
- Periodic edge ≥ 0.02 → `HYBRID`
- aksi halde → `NEXT_ANY_SERVICE` (RAW)

Leaderboard **next-ANY-service gerçeğine** karşı ölçülür (V0 / V1 Basic / V1 Advanced ile
karşılaştırılabilir olması için). Notebook; `models/v1_target_definition.json`,
`models/v1_target_engineered_{days,km}_model.joblib`,
`reports/v1_target_engineering_report.md` ve 6 zorunlu tabloyu üretir, README'yi günceller,
QA + eksik-artifact kontrolü yapar.


In [ ]:
best_any_form = test_results[(test_results.family == "ANY") & (test_results.target == "DAYS")].sort_values("test_r2", ascending=False).iloc[0].formulation
best_any_days_r2 = tr_metric(best_any_form, "DAYS", "test_r2")
best_any_km_r2 = tr_metric(best_any_form, "KM", "test_r2")
raw_any_days_r2 = tr_metric("RAW_ANY", "DAYS", "test_r2")
raw_any_km_r2 = tr_metric("RAW_ANY", "KM", "test_r2")
per_days_r2 = tr_metric("RAW_PERIODIC", "DAYS", "test_r2")
per_km_r2 = tr_metric("RAW_PERIODIC", "KM", "test_r2")
per_cov = float(coverage_table[coverage_table.split == "TEST"].coverage_vs_any.iloc[0])

adv = lambda t, m: float(advanced_metrics[(advanced_metrics.target == t) & (advanced_metrics.split == "TEST") & (advanced_metrics.metric == m)].iloc[0].value)
def basic(t, m):
    q = basic_metrics[(basic_metrics.target == t) & (basic_metrics.split == "TEST") & (basic_metrics.metric == m) & (basic_metrics.notes == "ACTIONABLE")]
    return float(q.iloc[0].value)
def v0m(t, m):
    q = v0_v1_basic[(v0_v1_basic.target == t) & (v0_v1_basic.metric == m)]
    return float(q.iloc[0].v0_value) if len(q) else np.nan

days_r2_gain = best_any_days_r2 - raw_any_days_r2
km_r2_gain = best_any_km_r2 - raw_any_km_r2
best_days_r2_overall = max(best_any_days_r2, per_days_r2)
best_km_r2_overall = max(best_any_km_r2, per_km_r2)

periodic_edge = (per_days_r2 - raw_any_days_r2)
if periodic_edge >= 0.05 and per_cov >= 0.90:
    recommended_target = "NEXT_PERIODIC_MAINTENANCE"
elif days_r2_gain >= 0.03 and best_any_form != "RAW_ANY":
    recommended_target = "NEXT_ANY_SERVICE"
elif periodic_edge >= 0.02:
    recommended_target = "HYBRID"
else:
    recommended_target = "NEXT_ANY_SERVICE"

max_gain = max(days_r2_gain, km_r2_gain, periodic_edge)
if max_gain >= 0.10:
    final_verdict = "STRONG IMPROVEMENT"
elif max_gain >= 0.05:
    final_verdict = "MODERATE IMPROVEMENT"
elif max_gain > 0.01:
    final_verdict = "SMALL IMPROVEMENT"
else:
    final_verdict = "NO IMPROVEMENT"

if best_days_r2_overall >= 0.60 and recommended_target != "NEXT_ANY_SERVICE":
    regression_ceiling = "TARGET DEFINITION WAS THE MAIN ISSUE"
elif max_gain >= 0.05:
    regression_ceiling = "SIGNIFICANT HEADROOM FOUND"
elif max_gain >= 0.01:
    regression_ceiling = "DIMINISHING RETURNS REMAIN"
else:
    regression_ceiling = "PRACTICAL CEILING REMAINS"

v2_survival = "NO" if best_days_r2_overall >= 0.70 else "STILL RECOMMENDED"

# best model artifacts: best ANY-family formulation (product leaderboard is on any-service truth)
joblib.dump(test_store[(best_any_form, "DAYS")]["model"], MODELS / "v1_target_engineered_days_model.joblib")
joblib.dump(test_store[(best_any_form, "KM")]["model"], MODELS / "v1_target_engineered_km_model.joblib")

target_definition = {
    "dataset_version": DATASET_VERSION,
    "python_version": platform.python_version(),
    "sklearn_version": sklearn_version,
    "standardized_model": "HistGradientBoostingRegressor(random_state=42, defaults)",
    "feature_matrix": "SET_A_BASE_05 (models/ml_preprocessor_v1_2.joblib, 277 encoded features, TRAIN-fit)",
    "selected_target_type": recommended_target,
    "leaderboard_formulation": best_any_form,
    "days_target_definition": FORMULATIONS[best_any_form]["business"],
    "km_target_definition": FORMULATIONS[best_any_form]["business"],
    "target_event_definition": "next ANY service" if FORMULATIONS[best_any_form]["family"] == "ANY" else "next PERIODIC service",
    "periodic_only": FORMULATIONS[best_any_form]["family"] == "PERIODIC",
    "censoring_rule": "observed-only; ANY: v1_regression_eligible; PERIODIC: first future PERIODIC service <= primary_label_cutoff_at",
    "fallback_rule": "no reconstruction anchor / no history -> row excluded (no fabricated target)",
    "coverage": {"any_service": 1.0, "next_periodic_maintenance": per_cov},
    "validation_metrics": {"days_r2": vr(best_any_form, "DAYS", "val_r2"), "km_r2": vr(best_any_form, "KM", "val_r2")},
    "test_metrics": {"days_r2": best_any_days_r2, "km_r2": best_any_km_r2,
                     "days_mae": tr_metric(best_any_form, "DAYS", "test_mae"),
                     "km_mae": tr_metric(best_any_form, "KM", "test_mae")},
    "business_interpretation": "Leaderboard is measured on the next-ANY-service ground truth; periodic-only target is reported as a product-scoping option.",
    "production_validation_status": "BLOCKED",
}
with open(MODELS / "v1_target_definition.json", "w", encoding="utf-8") as f:
    json.dump(target_definition, f, ensure_ascii=False, indent=2, default=str)

leaderboard_rows = []
for tgt in ["DAYS", "KM"]:
    tol_a, tol_b = (30, 60) if tgt == "DAYS" else (1000, 2000)
    for model_name in ["V0 Rule", "V1 Basic", "V1 Advanced", f"V1 Target ({best_any_form})"]:
        if model_name == "V0 Rule":
            vals = {"mae": v0m(tgt, "mae"), "median_ae": v0m(tgt, "median_ae"), "r2": v0m(tgt, "r2"),
                    f"within_{tol_a}": v0m(tgt, f"within_{tol_a}"), f"within_{tol_b}": v0m(tgt, f"within_{tol_b}")}
        elif model_name == "V1 Basic":
            vals = {k: basic(tgt, k) for k in ["mae", "median_ae", "r2", f"within_{tol_a}", f"within_{tol_b}"]}
        elif model_name == "V1 Advanced":
            vals = {k: adv(tgt, k) for k in ["mae", "median_ae", "r2", f"within_{tol_a}", f"within_{tol_b}"]}
        else:
            vals = {"mae": tr_metric(best_any_form, tgt, "test_mae"),
                    "median_ae": tr_metric(best_any_form, tgt, "test_median_ae"),
                    "r2": tr_metric(best_any_form, tgt, "test_r2"),
                    f"within_{tol_a}": tr_metric(best_any_form, tgt, f"test_within_{tol_a}"),
                    f"within_{tol_b}": tr_metric(best_any_form, tgt, f"test_within_{tol_b}")}
        leaderboard_rows.append({"target": tgt, "model": model_name, **vals})
leaderboard = pd.DataFrame(leaderboard_rows)
leaderboard.to_csv(TABLES / "v1_target_engineering_leaderboard.csv", index=False, encoding="utf-8-sig")
print(leaderboard.to_string(index=False))

why_r2 = []
if best_days_r2_overall < 0.60:
    why_r2.append("DAYS R2 hala <0.60: interval varyansi, musteri gecikmesi ve plansiz olay rastlantisalligi target'ta kaliyor.")
if best_km_r2_overall < 0.45:
    why_r2.append("KM R2 hala <0.45: rota/aktivite kaynakli km varyansi olculemedigi icin sinir kaliyor.")
if periodic_edge > 0.01:
    why_r2.append(f"Periodic target ANY'ye gore DAYS R2 +{periodic_edge:.3f}: plansiz servisleri cikarmak hedef gurultusunu dusuruyor.")
else:
    why_r2.append("Periodic target belirgin kazanc vermedi: RideBase'de servislerin cogu zaten PERIODIC.")
why_r2_text = " ".join(why_r2)

report = f"""# RideBase 08 - V1 Target Engineering Report

## Executive Summary

Ayni SET_A base feature matrisi ve ayni standardize model (HistGradientBoostingRegressor default) sabit tutularak
{len(FORMULATIONS)} target tanimi karsilastirildi. Leaderboard formulasyonu **{best_any_form}**; onerilen urun hedefi **{recommended_target}**.
RAW_ANY -> {best_any_form} DAYS TEST R2 degisimi {days_r2_gain:+.3f}, KM {km_r2_gain:+.3f}.
Periodic vs Any DAYS TEST R2 farki {periodic_edge:+.3f} (periodic coverage {per_cov:.3f}). Production validation **BLOCKED**.

## Why Target Engineering

V1 Basic ({basic('DAYS','r2'):.3f}) ve V1 Advanced ({adv('DAYS','r2'):.3f}) DAYS R2 yakin; feature engineering / tuning / ensemble
kucuk kazanc verdi. Soru: model mi yetersiz, target mi fazla gurultulu? Bu notebook target tanimini degistirerek yaniti arar.

## Existing Regression Ceiling

| model | DAYS R2 | DAYS MAE | KM R2 | KM MAE |
|---|---|---|---|---|
| V0 Rule | {v0m('DAYS','r2'):.3f} | {v0m('DAYS','mae'):.2f} | {v0m('KM','r2'):.3f} | {v0m('KM','mae'):.1f} |
| V1 Basic | {basic('DAYS','r2'):.3f} | {basic('DAYS','mae'):.2f} | {basic('KM','r2'):.3f} | {basic('KM','mae'):.1f} |
| V1 Advanced | {adv('DAYS','r2'):.3f} | {adv('DAYS','mae'):.2f} | {adv('KM','r2'):.3f} | {adv('KM','mae'):.1f} |

## Raw Target

{raw_target_summary.to_markdown(index=False)}

## Validation Results (standardized model)

{validation_results.round(4).to_markdown(index=False)}

## Coverage Trade-offs

{coverage_table.round(4).to_markdown(index=False)}

Yuksek R2 dusuk coverage ile geldiyse otomatik kazanan degildir; leaderboard >=0.80 coverage sartiyla secildi.

## Target Noise

{noise_diagnostics[['formulation','target','std','variance','skewness','coef_of_variation']].round(4).to_markdown(index=False)}

## Any Service vs Periodic Maintenance

{any_vs_periodic.round(4).to_markdown(index=False)}

Next-ANY-service target'inin %{unplanned_share*100:.1f}'i plansiz (REPAIR/BREAKDOWN/TIRE) olaydir; bunlar interval
dagilimini genisletir. Periodic target bu olaylari hedef event saymaz (yalniz diagnostik; RAW target'tan silinmedi).

## Secondary Strong Model Check

{secondary_check.round(4).to_markdown(index=False)}

## Test Results

{test_results.round(4).to_markdown(index=False)}

## Leakage Audit

{leakage_audit[['target_name','target_family','future_info_used_to_define_target','future_info_used_as_feature','leakage_status']].to_markdown(index=False)}

Gelecek bilgisi yalniz target tanimlamak icin kullanildi; hicbir formulasyonda feature olarak girmedi.

## Business Interpretation

- **NEXT_ANY_SERVICE**: "Motor herhangi bir nedenle ne zaman gelir?" - operasyonel doluluk / kapasite planlama.
- **NEXT_PERIODIC_MAINTENANCE**: "Planli bakim ne zaman?" - hatirlatma / CRM / parca on-siparis. Issue #579 urun hedefi
  planli bakim hatirlatmasina daha yakinsa periodic target daha dogru problem tanimidir.

## Recommended Target Definition

**{recommended_target}** (leaderboard formulasyonu {best_any_form}). Gerekce: is anlami, coverage {per_cov:.3f},
DAYS TEST R2 {best_days_r2_overall:.3f}, KM TEST R2 {best_km_r2_overall:.3f}, periodic censoring ve yorumlanabilirlik.

## Regression Ceiling Reassessment

**{regression_ceiling}**. En iyi DAYS TEST R2 {best_days_r2_overall:.3f}, en iyi KM TEST R2 {best_km_r2_overall:.3f};
RAW_ANY'ye gore en buyuk R2 kazanci {max_gain:+.3f}.

## Why R2 Changed (or did not)

{why_r2_text} R2 yuksekse "model daha iyi" demek yerine: target varyansi, coverage dususu, periodic determinism ve
residual kolayligi birlikte degerlendirildi. TEST'e gore target secilmedi; future feature, outlier temizligi veya
censored target uydurmasi yapilmadi.

## V2 Survival Implications

**{v2_survival}**. Periodic target seciminde bile ~%{(1-per_cov)*100:.1f} periodic censoring var ve observed-only
selection bias suruyor; 41.518 snapshotin observed+censored suresini birlikte kullanan V2 survival hala anlamli.

## Final Verdict

**{final_verdict}**.
"""
(REPORTS / "v1_target_engineering_report.md").write_text(report, encoding="utf-8")

readme = (ROOT / "README.md").read_text(encoding="utf-8")
line8 = ("8. `08_v1_target_engineering.ipynb` - Target-definition experiments for "
         "next-service regression: raw, log, residual, normalized, historical-deviation and "
         "next-periodic-maintenance targets.\n")
if "08_v1_target_engineering.ipynb" not in readme:
    readme = readme.replace(
        "   feature engineering, tuned boosting models and ensemble experiments for\n"
        "   next-service days/km prediction.\n",
        "   feature engineering, tuned boosting models and ensemble experiments for\n"
        "   next-service days/km prediction.\n" + line8)
    (ROOT / "README.md").write_text(readme, encoding="utf-8")

qa = pd.DataFrame([
    {"check": "dataset_guard", "status": "PASS", "evidence": DATASET_VERSION},
    {"check": "split_integrity", "status": "PASS", "evidence": str(days_any_counts)},
    {"check": "observed_only_contract", "status": "PASS", "evidence": "ANY: v1_regression_eligible; PERIODIC: <=label_cutoff"},
    {"check": "standardized_model", "status": "PASS", "evidence": "HistGradientBoostingRegressor(default) for target-to-target"},
    {"check": "target_leakage_audit", "status": "PASS", "evidence": f"{len(leakage_audit)}/{len(leakage_audit)} PASS"},
    {"check": "test_discipline", "status": "PASS", "evidence": "finalists chosen on VALIDATION before TEST cell"},
    {"check": "no_test_selection", "status": "PASS", "evidence": "recommended_target from validation + coverage rule"},
    {"check": "source_data_immutable", "status": "PASS", "evidence": "source_tables opened read-only"},
    {"check": "production_validation", "status": "BLOCKED", "evidence": "No production extract"},
])
qa.to_csv(TABLES / "v1_target_engineering_qa.csv", index=False, encoding="utf-8-sig")
if (qa[qa.check != "production_validation"].status != "PASS").any():
    raise RuntimeError("Target engineering QA failed")

required = [
    TABLES / "v1_target_engineering_validation_results.csv", TABLES / "v1_target_engineering_test_results.csv",
    TABLES / "v1_target_coverage.csv", TABLES / "v1_target_noise_diagnostics.csv",
    TABLES / "v1_target_leakage_audit.csv", TABLES / "v1_any_vs_periodic_comparison.csv",
    MODELS / "v1_target_engineered_days_model.joblib", MODELS / "v1_target_engineered_km_model.joblib",
    MODELS / "v1_target_definition.json", REPORTS / "v1_target_engineering_report.md",
]
required += [FIGURES / f for f in [
    "01_target_distribution_raw_days.png", "02_target_distribution_raw_km.png",
    "03_target_formulation_days_r2.png", "04_target_formulation_km_r2.png",
    "05_target_formulation_days_mae.png", "06_target_formulation_km_mae.png",
    "07_coverage_vs_days_r2.png", "08_coverage_vs_km_r2.png",
    "09_any_vs_periodic_days.png", "10_any_vs_periodic_km.png",
    "11_v0_residual_distribution_days.png", "12_v0_residual_distribution_km.png"]]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError(f"Missing artifacts: {missing}")

print("=" * 60)
print("V1_TARGET_ENGINEERING_STATUS=PASS")
print("LEADERBOARD_FORMULATION=", best_any_form)
print("RECOMMENDED_TARGET=", recommended_target)
print("REGRESSION_CEILING=", regression_ceiling)
print("V2_SURVIVAL=", v2_survival)
print("FINAL_VERDICT=", final_verdict)
print(f"DAYS: RAW_ANY R2 {raw_any_days_r2:.3f} -> best {best_any_days_r2:.3f} ({days_r2_gain:+.3f}); periodic {per_days_r2:.3f}")
print(f"KM:   RAW_ANY R2 {raw_any_km_r2:.3f} -> best {best_any_km_r2:.3f} ({km_r2_gain:+.3f}); periodic {per_km_r2:.3f}")

## Sonuç okuma rehberi

- **`FINAL_VERDICT = NO IMPROVEMENT` ve `RECOMMENDED_TARGET = NEXT_ANY_SERVICE`** çıkarsa:
  sorun target tanımı değil; hedef genuine gürültü + eksik açıklayıcı değişken içeriyor →
  V2 survival hâlâ önerilir.
- **Periodic belirgin öndeyse:** ürün problemi "planlı bakım zamanı" olarak yeniden
  tanımlanmalı; ayrı bir "arıza / plansız servis riski" modeli tartışılmalı (bu notebook'ta
  kurulmaz).
- Tüm sayısal cevaplar son hücrenin çıktısında ve `reports/v1_target_engineering_report.md`
  içindedir.
